# ✈️ TripMate – AI Travel Planner Telegram Bot
**Python + Telegram Bot API + Ollama (`llama3.2:1b`)**

This notebook runs the travel planner inside Telegram. You chat with your bot, and it asks 4 questions:
1. Destination
2. Budget, with a currency (e.g. `2000 MYR`)
3. Trip duration in days
4. Travel style, picked from buttons

It then sends you a full plan: a day-by-day itinerary, attractions, hotels, transport, a budget breakdown and tips. After that, you can keep chatting to adjust the plan.

**Commands:** `/start` or `/new` starts a new trip · `/save` sends the plan as a .txt file · `/cancel` · `/help`

---

## 🔧 Setup

**1. Ollama** (skip if already done)
- Install it from https://ollama.com/download
- Run `ollama pull llama3.2:1b` in a terminal
- Keep the Ollama app running, or run `ollama serve`

**2. Bot token**
Create a bot in @BotFather (`/newbot`, e.g. "TripMate") and paste its token into `BOT_TOKEN`. That keeps travel chats separate from fire alerts.

**3. Chat ID**
Your chat ID stays the same across bots. It belongs to your Telegram account, not to a bot. Here it's used as an allow-list so only you can use the bot. Set `ALLOWED_CHAT_IDS = set()` to let anyone use it.

**4. Run all cells**
Open your bot in Telegram, press **Start**, and follow the questions.

> ⚠️ The bot only responds **while the last cell is running** and your laptop is on. To stop it, press the ■ **Interrupt** button in Jupyter.

In [ ]:
%pip install ollama requests --quiet

## 1. Settings

In [ ]:
import math
import os
import re
import threading
import time

import ollama
import requests

# ===== Telegram =====
BOT_TOKEN = os.environ.get("TELEGRAM_BOT_TOKEN", "PASTE_YOUR_BOT_TOKEN_HERE")   # from @BotFather
ALLOWED_CHAT_IDS = set()   # e.g. {123456789} to make the bot private; set() = anyone
API = f"https://api.telegram.org/bot{BOT_TOKEN}"
TG_LIMIT = 4000                    # Telegram max is 4096 chars per message

# ===== Model =====
MODEL = "llama3.2:1b"
MAX_DAYS = 14

TRAVEL_STYLES = [
    "Budget / Backpacker",
    "Mid-range / Comfort",
    "Luxury",
    "Adventure & Outdoors",
    "Cultural & Historical",
    "Family-friendly",
    "Relaxation / Beach",
    "Food & Nightlife",
]

CURRENCY_ALIASES = {
    "usd": "USD", "$": "USD", "dollar": "USD", "dollars": "USD",
    "myr": "MYR", "rm": "MYR", "ringgit": "MYR",
    "eur": "EUR", "€": "EUR", "euro": "EUR", "euros": "EUR",
    "gbp": "GBP", "£": "GBP", "pound": "GBP", "pounds": "GBP",
    "pkr": "PKR", "rs": "PKR", "rupees": "PKR",
    "inr": "INR", "₹": "INR",
    "sar": "SAR", "riyal": "SAR", "riyals": "SAR",
    "aed": "AED", "dirham": "AED", "dirhams": "AED",
    "sgd": "SGD", "aud": "AUD",
}

SYSTEM_PROMPT = (
    "You are TripMate, a friendly and practical travel planning assistant. "
    "Give specific, realistic recommendations using real place names. "
    "Write in plain text: use the given section headings and simple '-' bullet points. "
    "Do NOT use markdown symbols like ** or #. "
    "Always mark prices as approximate estimates. "
    "For follow-up questions, answer briefly and directly based on the trip plan already discussed."
)

print("Settings loaded.")

## 2. Telegram helpers

In [ ]:
def tg(method, **params):
    """Call a Telegram Bot API method. Returns the JSON dict ({} on network failure)."""
    try:
        r = requests.post(f"{API}/{method}", json=params, timeout=40)
        data = r.json()
        if not data.get("ok"):
            print(f"Telegram {method} error:", data.get("description"))
        return data
    except (requests.RequestException, ValueError) as e:
        print(f"Telegram {method} failed:", e)
        return {}


def split_text(text, limit=TG_LIMIT):
    """Split long text into Telegram-sized chunks, preferring line breaks."""
    chunks = []
    while len(text) > limit:
        cut = text.rfind("\n", 0, limit)
        if cut < limit // 2:
            cut = limit
        chunks.append(text[:cut])
        text = text[cut:].lstrip("\n")
    if text.strip():
        chunks.append(text)
    return chunks


def style_keyboard():
    rows = [TRAVEL_STYLES[i:i + 2] for i in range(0, len(TRAVEL_STYLES), 2)]
    return {"keyboard": [[{"text": s} for s in row] for row in rows],
            "one_time_keyboard": True, "resize_keyboard": True}


def send_message(chat_id, text, keyboard=None, remove_keyboard=False):
    chunks = split_text(text) or ["(empty response)"]
    for i, chunk in enumerate(chunks):
        params = {"chat_id": chat_id, "text": chunk}
        if i == len(chunks) - 1:
            if keyboard:
                params["reply_markup"] = keyboard
            elif remove_keyboard:
                params["reply_markup"] = {"remove_keyboard": True}
        tg("sendMessage", **params)


def send_document(chat_id, text, filename="travel_plan.txt", caption=""):
    try:
        r = requests.post(f"{API}/sendDocument",
                          data={"chat_id": chat_id, "caption": caption},
                          files={"document": (filename, text.encode("utf-8"), "text/plain")},
                          timeout=40)
        if not r.json().get("ok"):
            print("sendDocument error:", r.text)
    except requests.RequestException as e:
        print("sendDocument failed:", e)


class Typing:
    """Keeps showing 'typing...' in Telegram while the model is generating."""
    def __init__(self, chat_id):
        self.chat_id = chat_id
        self.stop = threading.Event()

    def _loop(self):
        while not self.stop.is_set():
            tg("sendChatAction", chat_id=self.chat_id, action="typing")
            self.stop.wait(4)

    def __enter__(self):
        threading.Thread(target=self._loop, daemon=True).start()
        return self

    def __exit__(self, *exc):
        self.stop.set()


print("Telegram helpers ready.")

## 3. Ollama, validation and prompts

In [ ]:
def check_ollama(model=MODEL):
    try:
        resp = ollama.list()
    except Exception:
        return False, "Can't connect to Ollama. Open the Ollama app or run 'ollama serve'."
    models = resp["models"] if isinstance(resp, dict) else getattr(resp, "models", [])
    names = []
    for m in models:
        name = (m.get("model") or m.get("name")) if isinstance(m, dict) else getattr(m, "model", None)
        if name:
            names.append(name)
    if not any(n == model or n.startswith(model) for n in names):
        return False, f"Model '{model}' not found. Run: ollama pull {model}"
    return True, f"Connected to Ollama ({model})"


class ModelError(Exception):
    pass


def ask_model(messages):
    """Call Ollama and return the reply text, or raise ModelError with a user-friendly message."""
    try:
        resp = ollama.chat(model=MODEL, messages=messages,
                           options={"temperature": 0.7, "num_ctx": 4096, "num_predict": 2048})
        text = resp["message"]["content"].replace("**", "").strip()
        if not text:
            raise ModelError("The model returned an empty reply. Please try again.")
        return text
    except ollama.ResponseError as e:
        if getattr(e, "status_code", None) == 404:
            raise ModelError(f"Model '{MODEL}' isn't installed on the server. Run: ollama pull {MODEL}")
        raise ModelError(f"Ollama error: {getattr(e, 'error', e)}")
    except ConnectionError:
        raise ModelError("The AI server (Ollama) isn't running. Please start it and try again.")


# ---------- Validation (each returns (value, error)) ----------
def parse_destination(text):
    text = text.strip()
    if not 2 <= len(text) <= 80:
        return None, "Destination should be 2–80 characters. Try something like: Tokyo, Japan"
    if not any(ch.isalpha() for ch in text):
        return None, "That doesn't look like a place. Try something like: Bali, Indonesia"
    return text, None


def parse_budget(text):
    t = text.strip().lower()
    m = re.search(r"\d[\d,]*(?:\.\d+)?", t)
    if not m:
        return None, "Please send your budget as a number, e.g. 1500 USD or 2000 MYR"
    try:
        amount = float(m.group().replace(",", ""))
    except ValueError:
        return None, "I couldn't read that number. Try e.g. 1500 USD"
    if not math.isfinite(amount) or amount <= 0:
        return None, "Budget must be more than 0."
    if amount > 10_000_000:
        return None, "That budget looks too large. Please check it."
    rest = t[:m.start()] + " " + t[m.end():]
    currency = "USD"
    for token in re.findall(r"[a-z]+|[$€£₹]", rest):
        if token in CURRENCY_ALIASES:
            currency = CURRENCY_ALIASES[token]
            break
    return (amount, currency), None


def parse_days(text):
    m = re.search(r"\d+", text)
    if not m:
        return None, f"Please send the number of days (1–{MAX_DAYS})."
    days = int(m.group())
    if not 1 <= days <= MAX_DAYS:
        return None, f"Trip length must be between 1 and {MAX_DAYS} days."
    return days, None


def parse_style(text):
    t = text.strip().lower()
    for s in TRAVEL_STYLES:
        if t == s.lower():
            return s, None
    if t.isdigit() and 1 <= int(t) <= len(TRAVEL_STYLES):
        return TRAVEL_STYLES[int(t) - 1], None
    for s in TRAVEL_STYLES:               # partial match, e.g. "luxury", "beach"
        if t and any(t in part.strip().lower() for part in s.split("/")):
            return s, None
    return None, "Please pick a travel style from the buttons below 👇"


def build_prompt(destination, budget, currency, days, style):
    per_day = budget / days
    return f"""Create a personalised travel plan with these details:
- Destination: {destination}
- Trip length: {days} day(s)
- Total budget: {budget:,.0f} {currency} for the whole trip, for one traveller (about {per_day:,.0f} {currency} per day)
- Travel style: {style}

Use exactly these section headings, in this order:

1. TRIP OVERVIEW - 2-3 sentences on why this destination suits a {style.lower()} traveller.
2. DAY-BY-DAY ITINERARY - for every day from Day 1 to Day {days}, give Morning, Afternoon and Evening activities.
3. TOP ATTRACTIONS - 5 must-see places with one line each.
4. WHERE TO STAY - 3 hotel or accommodation options that fit the budget and style, with approximate price per night in {currency}.
5. TRANSPORT - how to get there and how to get around locally, with approximate costs in {currency}.
6. ESTIMATED BUDGET - breakdown for accommodation, food, transport, activities and extras in {currency}, with a total that stays within {budget:,.0f} {currency}.
7. TRAVEL TIPS - 5 practical tips (best time to visit, local customs, safety, money, packing).

If the budget is too low for this destination, say so honestly and suggest how to make it work."""


print(parse_budget("RM 2,000"), parse_budget("$1500"), parse_days("5 days"), parse_style("beach"))
ok, msg = check_ollama()
print(("✅ " if ok else "⚠️ ") + msg)

## 4. Bot conversation logic

In [ ]:
sessions = {}   # chat_id -> session dict

HELP_TEXT = (
    "🧭 TripMate commands\n\n"
    "/new – plan a new trip\n"
    "/save – get your plan as a .txt file\n"
    "/cancel – cancel the current planning\n"
    "/help – show this message\n\n"
    "Once your plan is ready, just type questions like:\n"
    "• Cheaper hotel options?\n"
    "• Swap day 2 for something outdoors\n"
    "• What food should I try?"
)


def new_session():
    return {"step": None, "trip": {}, "messages": [], "plan": ""}


def start_planning(chat_id):
    s = sessions[chat_id] = new_session()
    s["step"] = "destination"
    send_message(chat_id,
                 "👋 Hi! I'm TripMate, your AI travel planner.\n"
                 "I'll ask 4 quick questions, then build your trip plan.\n\n"
                 "1️⃣ Where do you want to go?\n(e.g. Langkawi, Malaysia)",
                 remove_keyboard=True)


def generate_plan(chat_id, s):
    t = s["trip"]
    send_message(chat_id,
                 f"🧳 Planning a {t['days']}-day {t['style']} trip to {t['destination']} "
                 f"with {t['budget']:,.0f} {t['currency']}...\n⏳ This can take up to a minute.",
                 remove_keyboard=True)
    s["messages"] = [{"role": "system", "content": SYSTEM_PROMPT},
                     {"role": "user", "content": build_prompt(**t)}]
    try:
        with Typing(chat_id):
            plan = ask_model(s["messages"])
    except ModelError as e:
        s["step"] = "style"
        send_message(chat_id, f"❌ {e}\n\nPick the style again to retry, or /new to start over.",
                     keyboard=style_keyboard())
        return
    s["messages"].append({"role": "assistant", "content": plan})
    s["plan"] = plan
    s["step"] = "chat"
    send_message(chat_id, plan)
    send_message(chat_id,
                 "✅ Your plan is ready! (Prices are AI estimates. Verify before booking.)\n\n"
                 "💬 Ask me anything to adjust it, /save for a .txt file, or /new for another trip.")


def answer_followup(chat_id, s, question):
    if len(question) > 500:
        send_message(chat_id, "Please keep questions under 500 characters.")
        return
    if len(s["messages"]) > 11:                     # keep context small for the 1B model
        s["messages"] = s["messages"][:3] + s["messages"][-8:]
    s["messages"].append({"role": "user", "content": question})
    try:
        with Typing(chat_id):
            reply = ask_model(s["messages"])
    except ModelError as e:
        s["messages"].pop()
        send_message(chat_id, f"❌ {e}")
        return
    s["messages"].append({"role": "assistant", "content": reply})
    s["plan"] += f"\n\n--- Q: {question} ---\n{reply}"
    send_message(chat_id, reply)


def handle_message(chat_id, text):
    text = text.strip()
    s = sessions.setdefault(chat_id, new_session())

    # ----- Commands -----
    if text.startswith("/"):
        cmd = text.split()[0].split("@")[0].lower()
        if cmd in ("/start", "/new"):
            start_planning(chat_id)
        elif cmd == "/help":
            send_message(chat_id, HELP_TEXT)
        elif cmd == "/cancel":
            sessions[chat_id] = new_session()
            send_message(chat_id, "❎ Cancelled. Send /new whenever you want to plan a trip.",
                         remove_keyboard=True)
        elif cmd == "/save":
            if s["plan"]:
                send_document(chat_id, s["plan"], caption="📄 Your TripMate travel plan")
            else:
                send_message(chat_id, "No plan yet. Send /new to create one.")
        else:
            send_message(chat_id, "Unknown command. Send /help to see what I can do.")
        return

    # ----- Conversation steps -----
    step = s["step"]
    if step is None:
        send_message(chat_id, "Hi! Send /new to plan a trip ✈️")

    elif step == "destination":
        value, err = parse_destination(text)
        if err:
            send_message(chat_id, "⚠️ " + err)
            return
        s["trip"]["destination"] = value
        s["step"] = "budget"
        send_message(chat_id, "2️⃣ What's your total budget for the trip?\n"
                              "Include the currency, e.g. 1500 USD or 2000 MYR (default is USD).")

    elif step == "budget":
        value, err = parse_budget(text)
        if err:
            send_message(chat_id, "⚠️ " + err)
            return
        s["trip"]["budget"], s["trip"]["currency"] = value
        s["step"] = "days"
        send_message(chat_id, f"3️⃣ How many days is the trip? (1–{MAX_DAYS})")

    elif step == "days":
        value, err = parse_days(text)
        if err:
            send_message(chat_id, "⚠️ " + err)
            return
        s["trip"]["days"] = value
        s["step"] = "style"
        send_message(chat_id, "4️⃣ Pick your travel style:", keyboard=style_keyboard())

    elif step == "style":
        value, err = parse_style(text)
        if err:
            send_message(chat_id, "⚠️ " + err, keyboard=style_keyboard())
            return
        s["trip"]["style"] = value
        generate_plan(chat_id, s)

    elif step == "chat":
        answer_followup(chat_id, s, text)


print("Bot logic ready.")

## 5. Run the bot 🚀
Run this cell, then message your bot in Telegram. Press the ■ **Interrupt** button to stop it.

In [ ]:
def run_bot():
    ok, msg = check_ollama()
    print(("✅ " if ok else "⚠️ ") + msg)

    me = tg("getMe")
    if not me.get("ok"):
        print("❌ Bot token is invalid. Check BOT_TOKEN.")
        return
    print(f"🤖 Bot @{me['result']['username']} is running. Open it in Telegram and send /start")
    print("   (Press ■ Interrupt in Jupyter to stop)\n")

    tg("deleteWebhook")                                   # make sure polling mode works
    offset = None
    old = tg("getUpdates", offset=-1, timeout=0)          # skip messages sent while offline
    if old.get("result"):
        offset = old["result"][-1]["update_id"] + 1

    try:
        while True:
            data = tg("getUpdates", offset=offset, timeout=30, allowed_updates=["message"])
            if not data.get("ok"):
                time.sleep(3)                             # network hiccup / conflict: back off
                continue
            for upd in data["result"]:
                offset = upd["update_id"] + 1
                msg = upd.get("message")
                if not msg:
                    continue
                chat_id = msg["chat"]["id"]
                if ALLOWED_CHAT_IDS and chat_id not in ALLOWED_CHAT_IDS:
                    send_message(chat_id, "🔒 Sorry, this is a private bot.")
                    print("Blocked chat:", chat_id)
                    continue
                text = msg.get("text")
                if not text:
                    send_message(chat_id, "Please send text messages only 🙂")
                    continue
                print(f"[{time.strftime('%H:%M:%S')}] {chat_id}: {text[:60]}")
                try:
                    handle_message(chat_id, text)
                except Exception as e:
                    print("Handler error:", type(e).__name__, e)
                    send_message(chat_id, "⚠️ Something went wrong. Please try again or send /new.")
    except KeyboardInterrupt:
        print("\n🛑 Bot stopped.")


run_bot()

## 🛠️ Troubleshooting

| Problem | Fix |
|---|---|
| Bot doesn't reply | Make sure the last cell is **still running** (shows `[*]`) and that you pressed Start in the bot chat |
| "Bot token is invalid" | Copy the token again from @BotFather. Paste it with no `[ ]` or `tel:` bits |
| "Conflict: terminated by other getUpdates request" | The bot is running in two places (two notebooks or cells). Stop one |
| "🔒 private bot" | Your chat ID isn't in `ALLOWED_CHAT_IDS`. Add it, or set `ALLOWED_CHAT_IDS = set()` |
| "Ollama isn't running" | Open the Ollama app or run `ollama serve` |
| "Model not installed" | Run `ollama pull llama3.2:1b` |
| Plans are generic or cut off | `llama3.2:1b` is small. Switch `MODEL` to `llama3.2:3b` (run `ollama pull llama3.2:3b` first) |

**Your fire-detection script still works with the same bot.** It only *sends* messages, so it doesn't conflict with this bot's polling. You'd just get fire alerts and travel chats in the same chat, which is why a separate TripMate bot is cleaner.